In [ ]:
# ---- 0. Install (Kaggle already has torch/numpy/matplotlib) ----
!pip install -q "monai>=1.3" nibabel scikit-image plotly gradio onnx onnxruntime tqdm joblib pandas nilearn

In [ ]:
# ---- 1. Imports & global configuration ----
import os, re, sys, json, time, copy, glob, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE}" +
      (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))


class CFG:
    # ---------------- run mode ----------------
    QUICK_MODE = False            # True -> 40 cases, 3 epochs each (pipeline sanity check)
    SEED = 42

    # ---------------- paths ----------------
    DATA_ROOT = None              # auto-detected if None (folder that contains BraTS20_Training_XXX)
    WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./neuroassist_out")
    CACHE_DIR = WORK_DIR / "cache"
    CKPT_DIR = WORK_DIR / "checkpoints"
    FIG_DIR = WORK_DIR / "figures"
    OUT_DIR = WORK_DIR / "outputs"

    # ---------------- data ----------------
    MODALITIES = ["flair", "t1", "t1ce", "t2"]
    CROP = (slice(56, 184), slice(56, 184), slice(13, 141))   # 240x240x155 -> 128^3 (brain-centred)
    VOL_SHAPE = (128, 128, 128)
    PATCH = (96, 96, 96)          # training patch (inference runs on the full 128^3 crop)
    VAL_FRACTION = 0.20
    N_CASES = None                # use only first N cases (debug)
    BATCH_SIZE = 2
    NUM_WORKERS = min(4, os.cpu_count() or 1)
    USE_ELASTIC = True            # elastic deformation augmentation (low probability)

    # ---------------- models ----------------
    TEACHER_FILTERS = (32, 64, 128, 256, 320)   # nnU-Net-style, residual blocks (the "SOTA-style" baseline)
    STUDENT_FILTERS = (8, 16, 32, 64, 128)      # lightweight
    # ---------------- training ----------------
    TEACHER_EPOCHS = 50
    STUDENT_EPOCHS = 50
    LR = 2e-4
    KD_ALPHA = 1.0                # weight of distillation loss
    KD_TEMP = 2.0                 # distillation temperature
    VAL_EVERY = 2
    USE_AMP = True
    RUN_SCRATCH_ABLATION = False  # also train the student WITHOUT distillation (shows KD benefit)

    # ---------------- extensions (all heavy ones are OFF by default) ----------------
    LONG_RUN = False              # True -> 150 epochs each ("train longer")
    RUN_CV = False                # 5-fold cross-validation of the lightweight model
    N_FOLDS = 5
    CV_EPOCHS = 30
    CV_WITH_KD = False            # also train a fold-specific teacher (5x teacher cost!)
    RUN_FEATURE_KD = False        # feature-level distillation variant
    FEAT_LAMBDA = 1.0
    RUN_REGION_HEAD = False       # sigmoid WT/TC/ET head variant
    RUN_DWSEP = True             # depthwise-separable student variant
    STUDENT_DW_FILTERS = (12, 24, 48, 96, 160)
    RUN_MC_DROPOUT_STUDENT = False  # student trained with dropout, for MC-dropout uncertainty
    STUDENT_DROPOUT = 0.1
    MC_SAMPLES = 10


if CFG.LONG_RUN:
    CFG.TEACHER_EPOCHS = CFG.STUDENT_EPOCHS = 150
if CFG.QUICK_MODE:
    CFG.N_CASES, CFG.TEACHER_EPOCHS, CFG.STUDENT_EPOCHS, CFG.VAL_EVERY = 40, 3, 3, 1
    CFG.CV_EPOCHS = 2

for d in (CFG.CACHE_DIR, CFG.CKPT_DIR, CFG.FIG_DIR, CFG.OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=CFG.SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()

# Region definitions (labels AFTER remapping 4 -> 3)
REGIONS = {"ET": (3,), "TC": (1, 3), "WT": (1, 2, 3)}
LABEL_NAMES = {1: "Necrotic / non-enhancing core", 2: "Peritumoral edema", 3: "Enhancing tumor"}
LABEL_COLORS = {1: (0.92, 0.18, 0.18), 2: (0.15, 0.80, 0.45), 3: (1.00, 0.85, 0.10)}   # red / green / yellow

In [ ]:
# ---- WHO ARE YOU? (switches on the experiments assigned to you) ----
WHO = "friend1"          # "friend1" or "friend2"

PRESETS = {"friend1": dict(RUN_FEATURE_KD=True, RUN_REGION_HEAD=True),
           "friend2": dict(RUN_DWSEP=True, RUN_MC_DROPOUT_STUDENT=True)}
for _k, _v in PRESETS[WHO].items():
    setattr(CFG, _k, _v)

# Laptop / small-GPU settings (uncomment if needed):
# CFG.NUM_WORKERS = 0      # Windows dataloader workers can hang
# CFG.BATCH_SIZE = 1       # if you run out of GPU memory
print("experiments on:", {k: getattr(CFG, k) for k in PRESETS[WHO]})

In [ ]:
import nibabel as nib
from joblib import Parallel, delayed


def scan_for_cases(base):
    """Walk `base` and return the first directory that directly contains BraTS20_Training_XXX folders."""
    if not base or not os.path.isdir(base):
        return None
    for dp, dn, _ in os.walk(base):
        if any(d.startswith("BraTS20_Training_") for d in dn):
            return dp
    return None


def find_data_root():
    """Robust against Kaggle's different mount layouts (/kaggle/input/<name>/... or /kaggle/input/datasets/<owner>/<name>/...)
    and against a DATA_ROOT that points one level too high (e.g. .../BraTS2020_TrainingData)."""
    cands = [CFG.DATA_ROOT,
             "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation",
             "/kaggle/input/brats20-dataset-training-validation"]
    for c in cands:
        hit = scan_for_cases(c)
        if hit:
            return hit
    for base in ["/kaggle/input", "/content", "/data", ".", str(CFG.WORK_DIR)]:
        hit = scan_for_cases(base)
        if hit:
            return hit
    try:  # Colab / local fallback (needs Kaggle credentials)
        import kagglehub
        hit = scan_for_cases(kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation"))
        if hit:
            return hit
    except Exception as e:
        print("kagglehub fallback failed:", e)
    raise FileNotFoundError("BraTS2020 not found. Add the Kaggle dataset or set CFG.DATA_ROOT to any folder above the cases.")


def find_case_files(case_dir):
    """Map modality -> file path (robust to the odd seg filename of case 355 in the Kaggle copy)."""
    files = {}
    for f in sorted(os.listdir(case_dir)):
        fl = f.lower()
        if not (fl.endswith(".nii") or fl.endswith(".nii.gz")):
            continue
        m = re.search(r"(flair|t1ce|t1gd|t1|t2)\.nii", fl)
        if m:
            files["t1ce" if m.group(1) == "t1gd" else m.group(1)] = os.path.join(case_dir, f)
        elif "seg" in fl:
            files["seg"] = os.path.join(case_dir, f)
    return files


DATA_ROOT = find_data_root()
ALL_IDS = sorted(d for d in os.listdir(DATA_ROOT)
                 if d.startswith("BraTS20_Training_") and os.path.isdir(os.path.join(DATA_ROOT, d)))
assert len(ALL_IDS) > 0, f"No BraTS20_Training_XXX folders in {DATA_ROOT}"
if CFG.N_CASES:
    ALL_IDS = ALL_IDS[:CFG.N_CASES]
print(f"Data root: {DATA_ROOT}\nCases found: {len(ALL_IDS)}")

In [ ]:
def load_nii(path):
    return nib.load(path).get_fdata(dtype=np.float32)


def zscore_nonzero(vol):
    """Clip to robust range, then z-score using non-zero (brain) voxels only; background stays 0."""
    mask = vol != 0
    if mask.sum() == 0:
        return vol
    v = vol[mask]
    lo, hi = np.percentile(v, [0.5, 99.5])
    v = np.clip(v, lo, hi)
    out = np.zeros_like(vol, dtype=np.float32)
    out[mask] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def preprocess_modalities(vols):
    """vols: dict modality -> (240,240,155) array  ->  (4,128,128,128) float32 (z-scored, cropped)."""
    chans = [zscore_nonzero(vols[m])[CFG.CROP] for m in CFG.MODALITIES]
    return np.stack(chans).astype(np.float32)


def preprocess_label(seg):
    seg = np.rint(seg).astype(np.uint8)[CFG.CROP]
    seg[seg == 4] = 3            # <-- label remap 4 -> 3 (contiguous classes 0,1,2,3)
    return seg


def cache_case(cid):
    ip, lp = CFG.CACHE_DIR / f"{cid}_img.npy", CFG.CACHE_DIR / f"{cid}_lbl.npy"
    if ip.exists() and lp.exists():
        return cid
    files = find_case_files(os.path.join(DATA_ROOT, cid))
    img = preprocess_modalities({m: load_nii(files[m]) for m in CFG.MODALITIES})
    lbl = preprocess_label(load_nii(files["seg"]))
    np.save(ip, img.astype(np.float16)); np.save(lp, lbl)
    return cid


def load_cached(cid):
    img = np.load(CFG.CACHE_DIR / f"{cid}_img.npy").astype(np.float32)
    lbl = np.load(CFG.CACHE_DIR / f"{cid}_lbl.npy")
    return img, lbl


t0 = time.time()
_ = Parallel(n_jobs=min(4, os.cpu_count() or 1))(delayed(cache_case)(c) for c in tqdm(ALL_IDS, desc="Caching"))
print(f"Cached {len(ALL_IDS)} cases in {time.time()-t0:.0f}s -> {CFG.CACHE_DIR}")

# Sanity checks
img0, lbl0 = load_cached(ALL_IDS[0])
print("image:", img0.shape, img0.dtype, "| label:", lbl0.shape, "| label values:", np.unique(lbl0))
assert set(np.unique(lbl0)).issubset({0, 1, 2, 3}), "label remap failed"

In [ ]:
# ---- Patient-level train / validation split ----
rng = np.random.RandomState(CFG.SEED)
perm = rng.permutation(len(ALL_IDS))
n_val = max(4, int(round(CFG.VAL_FRACTION * len(ALL_IDS))))
VAL_IDS = sorted(ALL_IDS[i] for i in perm[:n_val])
TRAIN_IDS = sorted(ALL_IDS[i] for i in perm[n_val:])
json.dump({"train": TRAIN_IDS, "val": VAL_IDS}, open(CFG.OUT_DIR / "split.json", "w"), indent=1)
print(f"train: {len(TRAIN_IDS)} cases | val: {len(VAL_IDS)} cases")

In [ ]:
from monai.transforms import (Compose, EnsureTyped, RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
                              RandAffined, Rand3DElasticd, RandScaleIntensityd, RandShiftIntensityd,
                              RandGaussianNoised)


def get_train_transform():
    keys = ["image", "label"]
    tf = [
        RandCropByPosNegLabeld(keys, label_key="label", spatial_size=CFG.PATCH, pos=3, neg=1, num_samples=1),
        RandFlipd(keys, prob=0.5, spatial_axis=0),
        RandFlipd(keys, prob=0.5, spatial_axis=1),
        RandFlipd(keys, prob=0.5, spatial_axis=2),
        RandRotate90d(keys, prob=0.3, max_k=3, spatial_axes=(0, 1)),
        RandAffined(keys, prob=0.25, rotate_range=(0.2, 0.2, 0.2), scale_range=(0.1, 0.1, 0.1),
                    mode=("bilinear", "nearest"), padding_mode="zeros"),
    ]
    if CFG.USE_ELASTIC:
        tf.append(Rand3DElasticd(keys, sigma_range=(5, 8), magnitude_range=(50, 100), prob=0.1,
                                 mode=("bilinear", "nearest"), padding_mode="zeros"))
    tf += [
        RandScaleIntensityd("image", factors=0.1, prob=0.5),
        RandShiftIntensityd("image", offsets=0.1, prob=0.5),
        RandGaussianNoised("image", prob=0.2, std=0.05),
        EnsureTyped(keys, dtype=torch.float32, track_meta=False),
    ]
    return Compose(tf)


def get_val_transform():
    return Compose([EnsureTyped(["image", "label"], dtype=torch.float32, track_meta=False)])


class BraTSCached(torch.utils.data.Dataset):
    def __init__(self, ids, transform):
        self.ids, self.transform = ids, transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img, lbl = load_cached(self.ids[i])
        out = self.transform({"image": img, "label": lbl[None].astype(np.float32)})
        return out[0] if isinstance(out, (list, tuple)) else out     # RandCropByPosNegLabeld returns a list


def _worker_init(worker_id):
    """Give every dataloader worker a different augmentation RNG stream."""
    seed = torch.initial_seed() % (2 ** 32)
    ds = torch.utils.data.get_worker_info().dataset
    ds.transform.set_random_state(seed=seed)
    np.random.seed(seed); random.seed(seed)


def make_loaders(train_ids=None, val_ids=None):
    tr = BraTSCached(train_ids or TRAIN_IDS, get_train_transform())
    va = BraTSCached(val_ids or VAL_IDS, get_val_transform())
    nw = CFG.NUM_WORKERS
    kw = dict(num_workers=nw, pin_memory=DEVICE.type == "cuda", worker_init_fn=_worker_init if nw else None)
    train_loader = torch.utils.data.DataLoader(tr, batch_size=CFG.BATCH_SIZE, shuffle=True, drop_last=True,
                                               persistent_workers=nw > 0, **kw)
    val_loader = torch.utils.data.DataLoader(va, batch_size=1, shuffle=False, **kw)
    return train_loader, val_loader


train_loader, val_loader = make_loaders()
b = next(iter(train_loader))
print("train batch:", b["image"].shape, b["label"].shape, "| label values:", torch.unique(b["label"]).tolist())

In [ ]:
from monai.networks.nets import DynUNet
from monai.losses import DiceCELoss


def build_model(filters, res_block, out_ch=4, dropout=None):
    n = len(filters)
    return DynUNet(spatial_dims=3, in_channels=4, out_channels=out_ch,
                   kernel_size=[3] * n, strides=[1] + [2] * (n - 1), upsample_kernel_size=[2] * (n - 1),
                   filters=list(filters), res_block=res_block, deep_supervision=False, dropout=dropout)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_teacher():
    return build_model(CFG.TEACHER_FILTERS, res_block=True)


def build_student():
    return build_model(CFG.STUDENT_FILTERS, res_block=False)


_t, _s = build_teacher(), build_student()
print(f"Baseline (teacher): {count_params(_t)/1e6:6.2f} M params")
print(f"Lightweight (student): {count_params(_s)/1e6:6.2f} M params  "
      f"(-> {count_params(_t)/count_params(_s):.1f}x fewer)")
del _t, _s

In [ ]:
seg_loss_fn = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False, batch=True)


def kd_loss_fn(s_logits, t_logits, T):
    s = F.log_softmax(s_logits / T, dim=1)
    t = F.softmax(t_logits / T, dim=1)
    return F.kl_div(s, t, reduction="none").sum(1).mean() * (T * T)


def region_dice_torch(pred, gt):
    """pred, gt: integer label volumes (same shape) -> dict region -> dice (1.0 if both empty)."""
    out = {}
    for r, ids in REGIONS.items():
        p = torch.zeros_like(pred, dtype=torch.bool); g = torch.zeros_like(gt, dtype=torch.bool)
        for i in ids:
            p |= pred == i; g |= gt == i
        s = p.sum().item() + g.sum().item()
        out[r] = 1.0 if s == 0 else 2.0 * (p & g).sum().item() / s
    return out


@torch.no_grad()
def validate(model, loader):
    model.eval()
    scores = {r: [] for r in REGIONS}
    for b in loader:
        x = b["image"].to(DEVICE, non_blocking=True)
        y = b["label"].long().to(DEVICE).squeeze(1)
        with torch.autocast(DEVICE.type, enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
            logits = model(x)
        pred = logits.argmax(1)
        for k in range(x.shape[0]):
            d = region_dice_torch(pred[k], y[k])
            for r in REGIONS:
                scores[r].append(d[r])
    res = {r: float(np.mean(v)) for r, v in scores.items()}
    res["mean"] = float(np.mean(list(res.values())))
    return res

In [ ]:
def train_model(name, model, train_loader, val_loader, epochs, lr=None, teacher=None,
                kd_alpha=0.0, kd_T=2.0, resume=True, feat_distill=None, feat_lambda=1.0):
    """Generic trainer. If `teacher` is given, adds the distillation loss. Returns (best model, history df)."""
    lr = lr or CFG.LR
    use_amp = CFG.USE_AMP and DEVICE.type == "cuda"
    model.to(DEVICE)
    best_p, last_p = CFG.CKPT_DIR / f"{name}_best.pt", CFG.CKPT_DIR / f"{name}_last.pt"
    params = list(model.parameters()) + (list(feat_distill.adapters.parameters()) if feat_distill is not None else [])
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    hist, best, start = [], -1.0, 0

    if resume and last_p.exists():
        ck = torch.load(last_p, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); hist, best, start = ck["hist"], ck["best"], ck["epoch"] + 1
        print(f"[{name}] resumed from epoch {start}")

    if teacher is not None:
        teacher.to(DEVICE).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)

    for ep in range(start, epochs):
        model.train(); t0 = time.time(); run, n = 0.0, 0
        pbar = tqdm(train_loader, leave=False, desc=f"{name} {ep+1}/{epochs}")
        if feat_distill is not None:
            feat_distill.enabled = True
        for b in pbar:
            x = b["image"].to(DEVICE, non_blocking=True)
            y = b["label"].long().to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=use_amp):
                logits = model(x)
                loss = seg_loss_fn(logits.float(), y)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(x)
                    loss = loss + kd_alpha * kd_loss_fn(logits.float(), t_logits.float(), kd_T)
                    if feat_distill is not None:      # feature-level distillation (encoder features)
                        loss = loss + feat_lambda * feat_distill.loss()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0)
            scaler.step(opt); scaler.update()
            run += loss.item(); n += 1
            pbar.set_postfix(loss=f"{run/n:.4f}")
        sched.step()
        if feat_distill is not None:
            feat_distill.enabled = False

        row = {"epoch": ep + 1, "train_loss": run / max(n, 1), "lr": opt.param_groups[0]["lr"],
               "time_s": time.time() - t0, "val_mean": np.nan, "val_ET": np.nan, "val_TC": np.nan, "val_WT": np.nan}
        if (ep + 1) % CFG.VAL_EVERY == 0 or ep + 1 == epochs:
            v = validate(model, val_loader)
            row.update(val_mean=v["mean"], val_ET=v["ET"], val_TC=v["TC"], val_WT=v["WT"])
            if v["mean"] > best:
                best = v["mean"]
                torch.save({"model": model.state_dict(), "epoch": ep, "val": v}, best_p)
            print(f"[{name}] ep {ep+1:3d}  loss {row['train_loss']:.4f}  "
                  f"Dice ET {v['ET']:.3f} TC {v['TC']:.3f} WT {v['WT']:.3f} | mean {v['mean']:.3f}  "
                  f"(best {best:.3f})  {row['time_s']:.0f}s")
        hist.append(row)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "hist": hist, "best": best, "epoch": ep}, last_p)

    if best_p.exists():
        model.load_state_dict(torch.load(best_p, map_location=DEVICE, weights_only=False)["model"])
    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(CFG.OUT_DIR / f"history_{name}.csv", index=False)
    return model, hist_df

In [ ]:
# ---- Load the shared weights (no retraining) ----
def find_file(fname):
    cands = [CFG.CKPT_DIR / fname, Path("checkpoints") / fname, Path(fname)]
    cands += [Path(p) for p in glob.glob(f"/kaggle/input/**/{fname}", recursive=True)]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError(f"{fname} not found. Put it in ./checkpoints/ or add the Kaggle dataset shared by the lead.")


def load_pretrained(model, fname):
    p = find_file(fname)
    ck = torch.load(p, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    print(f"loaded {p}")
    return model.to(DEVICE).eval()


teacher = load_pretrained(build_teacher(), "teacher_best.pt")
student = load_pretrained(build_student(), "student_kd_best.pt")
student_scratch = None

# Sanity check: must match the lead's numbers (teacher ET 0.761 / TC 0.834 / WT 0.903, mean 0.833)
print("teacher :", {k: round(v, 3) for k, v in validate(teacher, val_loader).items()})
print("student :", {k: round(v, 3) for k, v in validate(student, val_loader).items()})

In [ ]:
import copy


@torch.no_grad()
def bench(model, device, half=False, shape=(1, 4, 128, 128, 128), warm=3, runs=10):
    m = copy.deepcopy(model).to(device).eval()
    x = torch.randn(*shape, device=device)
    if half:
        m, x = m.half(), x.half()
    if device.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    for _ in range(warm):
        m(x)
    ts = []
    for _ in range(runs):
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter(); m(x)
        if device.type == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    mem = torch.cuda.max_memory_allocated() / 2 ** 20 if device.type == "cuda" else np.nan
    del m, x
    return float(np.mean(ts)), float(np.std(ts)), mem

In [ ]:
EXT = {}          # name -> markdown table (goes to README)
VARIANTS = []     # one row per trained model variant


def df_to_md(df, floatfmt="{:.3f}"):
    cols = [df.index.name or "model"] + list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for idx, row in df.iterrows():
        out.append("| " + " | ".join([str(idx)] + [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v)
                                                    for v in row.values]) + " |")
    return "\n".join(out)


def record_variant(name, model, val=None):
    """Add a row (params, latency, Dice) to the variants table."""
    val = val or validate(model, val_loader)
    dev, half = (DEVICE, True) if DEVICE.type == "cuda" else (torch.device("cpu"), False)
    ms = bench(model, dev, half=half, warm=1, runs=3)[0]
    VARIANTS.append({"variant": name, "Params (M)": round(count_params(model) / 1e6, 2), "latency (ms)": round(ms, 1),
                     "Dice ET": round(val["ET"], 3), "Dice TC": round(val["TC"], 3), "Dice WT": round(val["WT"], 3),
                     "Dice mean": round(val["mean"], 3)})


record_variant("Baseline (teacher)", teacher)
record_variant("Lightweight + logit KD", student)
if student_scratch is not None:
    record_variant("Lightweight, no KD", student_scratch)
pd.DataFrame(VARIANTS)

In [ ]:
class FeatureDistiller(nn.Module):
    """Forward hooks on the 5 encoder stages of student & teacher (DynUNet: input_block, downsamples, bottleneck)."""

    def __init__(self, student, teacher):
        super().__init__()
        enc = lambda m: [m.input_block, *m.downsamples, m.bottleneck]
        self.s_feats, self.t_feats, self.enabled, self.handles = {}, {}, False, []
        s_mods, t_mods = enc(student), enc(teacher)
        assert len(s_mods) == len(t_mods) == len(CFG.STUDENT_FILTERS)
        for i, (sm, tm) in enumerate(zip(s_mods, t_mods)):
            self.handles.append(sm.register_forward_hook(lambda m, a, o, i=i: self.s_feats.__setitem__(i, o) if self.enabled else None))
            self.handles.append(tm.register_forward_hook(lambda m, a, o, i=i: self.t_feats.__setitem__(i, o) if self.enabled else None))
        self.adapters = nn.ModuleList([nn.Conv3d(s, t, 1) for s, t in zip(CFG.STUDENT_FILTERS, CFG.TEACHER_FILTERS)])

    def loss(self):
        tot = 0.0
        for i, a in enumerate(self.adapters):
            fs = F.normalize(a(self.s_feats[i]).float(), dim=1)
            ft = F.normalize(self.t_feats[i].float().detach(), dim=1)
            tot = tot + F.mse_loss(fs, ft)
        self.s_feats.clear(); self.t_feats.clear()
        return tot / len(self.adapters)

    def close(self):
        for h in self.handles:
            h.remove()


student_fd = None
if CFG.RUN_FEATURE_KD:
    student_fd = build_student()
    fd = FeatureDistiller(student_fd, teacher).to(DEVICE)
    student_fd, hist_fd = train_model("student_featkd", student_fd, train_loader, val_loader, CFG.STUDENT_EPOCHS,
                                      teacher=teacher, kd_alpha=CFG.KD_ALPHA, kd_T=CFG.KD_TEMP,
                                      feat_distill=fd, feat_lambda=CFG.FEAT_LAMBDA)
    fd.close()
    record_variant("Lightweight + logit KD + feature KD", student_fd)
else:
    print("Skipped (CFG.RUN_FEATURE_KD = False)")

In [ ]:
from monai.losses import DiceLoss


def regions_from_labels(y):                       # (B,1,...) long -> (B,3,...) float [WT, TC, ET]
    y = y[:, 0]
    return torch.stack([y > 0, (y == 1) | (y == 3), y == 3], 1).float()


def decode_regions(prob):                         # (B,3,...) sigmoid probs -> (B,...) label volume (0,1,2,3)
    wt, tc, et = prob[:, 0] > .5, prob[:, 1] > .5, prob[:, 2] > .5
    lab = torch.zeros_like(wt, dtype=torch.long)
    lab[wt] = 2
    lab[wt & tc] = 1
    lab[wt & tc & et] = 3
    return lab


def train_region_student(epochs):
    model = build_model(CFG.STUDENT_FILTERS, False, out_ch=3).to(DEVICE)
    dice_l, use_amp = DiceLoss(sigmoid=True, batch=True), CFG.USE_AMP and DEVICE.type == "cuda"
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=CFG.LR * 0.02)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    best, best_sd = -1, None
    for ep in range(epochs):
        model.train()
        for b in tqdm(train_loader, leave=False, desc=f"region-head {ep+1}/{epochs}"):
            x, y = b["image"].to(DEVICE), regions_from_labels(b["label"].long().to(DEVICE))
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=use_amp):
                z = model(x).float()
                loss = dice_l(z, y) + F.binary_cross_entropy_with_logits(z, y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0); scaler.step(opt); scaler.update()
        sched.step()
        if (ep + 1) % CFG.VAL_EVERY == 0 or ep + 1 == epochs:
            model.eval(); sc = {r: [] for r in REGIONS}
            with torch.no_grad():
                for b in val_loader:
                    with torch.autocast(DEVICE.type, enabled=use_amp):
                        z = model(b["image"].to(DEVICE))
                    pred = decode_regions(torch.sigmoid(z.float()))
                    d = region_dice_torch(pred[0], b["label"].long().to(DEVICE)[0, 0])
                    for r in REGIONS: sc[r].append(d[r])
            v = {r: float(np.mean(s)) for r, s in sc.items()}; v["mean"] = float(np.mean(list(v.values())))
            print(f"[region-head] ep {ep+1} Dice ET {v['ET']:.3f} TC {v['TC']:.3f} WT {v['WT']:.3f} mean {v['mean']:.3f}")
            if v["mean"] > best:
                best, best_v, best_sd = v["mean"], v, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_sd)
    return model, best_v


if CFG.RUN_REGION_HEAD:
    student_region, v_region = train_region_student(CFG.STUDENT_EPOCHS)
    record_variant("Lightweight, sigmoid region head (no KD)", student_region, v_region)
else:
    print("Skipped (CFG.RUN_REGION_HEAD = False)")

In [ ]:
def to_depthwise_separable(model):
    """Replace every 3x3x3 Conv3d by depthwise(3x3x3, groups=C_in) + pointwise(1x1x1)."""
    for name, ch in list(model.named_children()):
        if isinstance(ch, nn.Conv3d) and ch.kernel_size == (3, 3, 3):
            dw = nn.Conv3d(ch.in_channels, ch.in_channels, 3, stride=ch.stride, padding=ch.padding,
                           groups=ch.in_channels, bias=False)
            pw = nn.Conv3d(ch.in_channels, ch.out_channels, 1, bias=ch.bias is not None)
            setattr(model, name, nn.Sequential(dw, pw))
        else:
            to_depthwise_separable(ch)
    return model


def build_student_dw():
    return to_depthwise_separable(build_model(CFG.STUDENT_DW_FILTERS, False))


m_dw = build_student_dw()
print(f"Depthwise-separable student: {count_params(m_dw)/1e6:.2f} M params (plain student: {count_params(student)/1e6:.2f} M)")
if CFG.RUN_DWSEP:
    student_dw, _ = train_model("student_dw", m_dw, train_loader, val_loader, CFG.STUDENT_EPOCHS, teacher=teacher,
                                kd_alpha=CFG.KD_ALPHA, kd_T=CFG.KD_TEMP)
    record_variant("Depthwise-separable student + KD", student_dw)
else:
    print("Training skipped (CFG.RUN_DWSEP = False)")

In [ ]:
# ---- Student trained with dropout (for MC-dropout uncertainty) ----
student_mc = None
if CFG.RUN_MC_DROPOUT_STUDENT:
    student_mc = build_model(CFG.STUDENT_FILTERS, False, dropout=CFG.STUDENT_DROPOUT)
    student_mc, _ = train_model("student_mc", student_mc, train_loader, val_loader, CFG.STUDENT_EPOCHS, teacher=teacher,
                                kd_alpha=CFG.KD_ALPHA, kd_T=CFG.KD_TEMP)
    record_variant(f"Lightweight + KD + dropout {CFG.STUDENT_DROPOUT}", student_mc)
else:
    print("Skipped (CFG.RUN_MC_DROPOUT_STUDENT = False)")

In [ ]:
# ---- Variants comparison (everything trained so far) ----
variants_df = pd.DataFrame(VARIANTS).set_index("variant")
display(variants_df)
EXT["Model variants (validation Dice; latency = GPU FP16 or CPU, 128³)"] = df_to_md(variants_df, "{:.3f}")
variants_df.to_csv(CFG.OUT_DIR / "variants.csv")

In [ ]:
# ---- Package results for the lead ----
import shutil
R = CFG.OUT_DIR / f"{WHO}_results"
R.mkdir(parents=True, exist_ok=True)
pd.DataFrame(VARIANTS).to_csv(R / "variants.csv", index=False)
for f in CFG.OUT_DIR.glob("history_*.csv"):
    shutil.copy(f, R / f.name)
for f in CFG.CKPT_DIR.glob("*_best.pt"):
    if f.name not in ("teacher_best.pt", "student_kd_best.pt"):
        shutil.copy(f, R / f.name)
for f in CFG.FIG_DIR.glob("*.png"):
    shutil.copy(f, R / f.name)
z = shutil.make_archive(str(CFG.OUT_DIR / "friend_results"), "zip", R)
print("Send this file to the lead:", z, f"({Path(z).stat().st_size/2**20:.0f} MB)")
display(pd.DataFrame(VARIANTS))